# Why MILP, and not the genetic algorithm

**Chapter 7 evidence — solver selection for warehouse slotting**

This notebook answers one question: *given two candidate search strategies, on what
grounds was mixed-integer linear programming chosen over a genetic algorithm?*

It is deliberately not a claim that MILP is universally superior. The result below
shows the two methods are **indistinguishable on small instances**, and the case for
MILP rests on how that changes with problem size, plus a guarantee the GA cannot
provide at any size.

## 1. Why the obvious comparison would have been invalid

The project contains a working GA (`slotting-service/app/api/`) and a working MILP
(`slotting-service/app/services/plan_optimizer.py`). Benchmarking them against each
other as they ship would have been meaningless, because they do not solve the same
problem:

| | Shipped GA | Shipped MILP |
|---|---|---|
| Decision | one parcel at a time, sequentially | all materials simultaneously |
| Location space | own encoding, `A-01-01-1-A` (2-digit slot) | v8 warehouse pool, `A-01-001-1-A` (3-digit bay) |
| Objective | 12 weighted placement terms | travel + accessibility + vertical + relocation |
| Feasible set | its own `WarehouseState` | 4,206 real v8 pick faces |

Their location codes are not even mutually decodable. A head-to-head between them
would compare **problems**, not **methods**, and any difference would be
uninterpretable.

### The fix

Both strategies are run here over **one identical formulation**:

- the same decision variables (one pick face per material),
- the same candidate sets, generated by the shipped solver's own
  `_milp_candidate_locations`,
- the same constraints (one material per pick face, relocation cap),
- and **one shared objective function** that scores whatever assignment either
  method returns.

The GA in this notebook is therefore not the shipped per-parcel GA. It is a standard
generational GA — tournament selection, uniform crossover, elitism, and repair of
double-booked pick faces — written against the MILP's own formulation. Any remaining
difference is attributable to the search strategy alone.

## 2. Experimental design

| Factor | Setting | Reason |
|---|---|---|
| Problem sizes | 20, 40, 80, 120 materials | to detect scaling behaviour, not a single point |
| Instances per size | 5 independent random subsets | gives paired replicates for a within-instance test |
| GA seeds per instance | 5 | the GA is stochastic; one run would not characterise it |
| GA compute budget | ≥ 5× the MILP's own wall-clock | so the GA cannot lose merely by being given less time |
| MILP time limit | 120 s (never approached) | |

**Pairing.** Each instance is solved by both methods, so observations are paired and
the between-instance variation in cost — which is large, since instances differ in
size — is removed from the comparison.

**Tests.** Wilcoxon signed-rank (one-sided), which assumes neither normality nor
equal variances and is appropriate for n = 20 paired observations on a skewed cost
scale. Effect size is the matched-pairs rank-biserial correlation. Confidence
intervals on the optimality gap are percentile bootstrap, assuming no parametric
form. Scaling is tested with Spearman's rank correlation.

In [1]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, ".")

import numpy as np, pandas as pd
from scipy import stats
from pipeline.solver_comparison import (
    bootstrap_ci, cliffs_delta, matched_pairs_rank_biserial,
)

df = pd.read_csv("outputs/solver_evidence/solver_comparison_rows.csv")
print(f"{len(df)} paired observations over sizes {sorted(df['size'].unique())}")
print(f"GA seeds per instance: {int(df.ga_runs.iloc[0])}")
df[["size","instance","milp_cost","milp_seconds","ga_best","ga_mean","ga_sd","ga_budget"]].round(2)

20 paired observations over sizes [np.int64(20), np.int64(40), np.int64(80), np.int64(120)]
GA seeds per instance: 5


,size,instance,milp_cost,milp_seconds,ga_best,ga_mean,ga_sd,ga_budget
0,20,0,1229.89,0.01,1229.89,1244.72,21.56,5.0
1,20,1,1982.50,0.01,1982.50,2011.63,19.89,5.0
2,20,2,1614.24,0.01,1614.24,1614.53,0.65,5.0
3,20,3,1373.13,0.01,1373.13,1379.33,8.83,5.0
4,20,4,1283.82,0.01,1314.73,1335.81,17.56,5.0
5,40,0,3443.55,0.01,3528.00,3580.35,52.60,5.0
6,40,1,4066.41,0.02,4098.72,4198.64,99.27,5.0
7,40,2,3352.22,0.03,3445.64,3484.14,35.32,5.0
8,40,3,2737.88,0.02,2893.09,2991.68,60.82,5.0
9,40,4,3459.30,0.02,3537.49,3586.15,79.52,5.0


## 3. The headline result: the two methods are not separable at small size

Reported as an **optimality gap** — how far the GA's best-of-5 lands above the MILP
cost, as a percentage. Because the MILP solution is *proven* optimal on every
instance (§5), this gap is a true distance from the global optimum, not a relative
score between two heuristics.

In [2]:
by_size = df.groupby("size").apply(lambda g: pd.Series({
    "MILP ms":            g.milp_seconds.mean() * 1000,
    "GA budget s":        g.ga_budget.mean(),
    "gap best %":         g.gap_best_pct.mean(),
    "gap mean %":         g.gap_mean_pct.mean(),
    "CI low":             bootstrap_ci(g.gap_best_pct.values, resamples=4000)[0],
    "CI high":            bootstrap_ci(g.gap_best_pct.values, resamples=4000)[1],
    "GA run-to-run CV %": g.ga_cv_pct.mean(),
}), include_groups=False).round(2)
by_size

,MILP ms,GA budget s,gap best %,gap mean %,CI low,CI high,GA run-to-run CV %
size,,,,,,,
20,8.66,5.0,0.48,1.44,-0.00,1.44,0.94
40,19.59,5.0,2.79,4.82,1.49,4.34,1.82
80,50.31,5.0,20.33,24.85,18.09,22.57,3.06
120,101.90,5.0,26.40,29.71,25.59,27.15,1.95


At **n = 20 the confidence interval includes zero** — on problems this small the GA
finds the optimum and there is no evidence to prefer either method on solution
quality. That is stated plainly because it bounds the claim: the argument for MILP is
not that it always wins.

From n = 80 the picture changes entirely.

In [3]:
w = stats.wilcoxon(df.ga_best, df.milp_cost, alternative="greater")
r, r_label = matched_pairs_rank_biserial(df.ga_best.values, df.milp_cost.values)
lo, hi = bootstrap_ci(df.gap_best_pct.values)
rho = stats.spearmanr(df["size"], df.gap_best_pct)

print("Paired comparison over all 20 instances (GA best-of-5 vs MILP)")
print(f"  Wilcoxon signed-rank  W = {w.statistic:.0f},  p = {w.pvalue:.2e}")
print(f"  matched-pairs rank-biserial r = {r:+.3f} ({r_label})")
print(f"  mean optimality gap = {df.gap_best_pct.mean():.2f}%  95% CI [{lo:.2f}, {hi:.2f}]")
print(f"  instances where the GA beat the MILP: {(df.ga_best < df.milp_cost - 1e-9).sum()} of {len(df)}")
print()
print("Does the gap grow with problem size?")
print(f"  Spearman rho = {rho.statistic:+.3f},  p = {rho.pvalue:.2e}")

Paired comparison over all 20 instances (GA best-of-5 vs MILP)
  Wilcoxon signed-rank  W = 200,  p = 1.95e-04
  matched-pairs rank-biserial r = +0.905 (large)
  mean optimality gap = 12.50%  95% CI [7.72, 17.44]
  instances where the GA beat the MILP: 0 of 20

Does the gap grow with problem size?
  Spearman rho = +0.954,  p = 7.48e-11


### Reading these numbers

- **p = 1.9e-04** rejects the null that the two methods produce equal cost. With 20
  paired observations, Wilcoxon is the right test — a paired *t*-test would assume a
  normality that the gap distribution does not have.
- **r = +0.905** is the effect size *on the paired differences*. This matters: the
  unpaired Cliff's delta on the same data returns +0.140 ("negligible"), because
  pooling instances of different sizes lets between-instance cost variation swamp the
  within-instance effect. Reporting that number here would be a category error, and
  it is shown below only to be discounted.
- **rho = +0.954, p = 7.5e-11** is the load-bearing result. The disadvantage is not a
  fixed offset; it grows monotonically with problem size, and the real warehouse sits
  at the top of the tested range.

In [4]:
d_unpaired, d_label = cliffs_delta(df.ga_best.values, df.milp_cost.values)
print(f"Unpaired Cliff's delta on the same data: {d_unpaired:+.3f} ({d_label})")
print("-> wrong statistic for a paired design; retained only to show the discrepancy.")

Unpaired Cliff's delta on the same data: +0.140 (negligible)
-> wrong statistic for a paired design; retained only to show the discrepancy.


## 4. Compute cost

The GA was given far more time than the MILP used and still lost.

In [5]:
print(f"MILP median solve time : {df.milp_seconds.median()*1000:8.1f} ms")
print(f"GA budget per seed     : {df.ga_budget.median():8.1f} s")
print(f"GA advantage in compute: {df.ga_budget.median()/df.milp_seconds.median():8.0f}x")
print(f"MILP at the largest size tested (n=120): {df[df['size']==120].milp_seconds.mean()*1000:.0f} ms")

MILP median solve time :     36.3 ms
GA budget per seed     :      5.0 s
GA advantage in compute:      138x
MILP at the largest size tested (n=120): 102 ms


## 5. The decisive argument is not statistical

Every result above is a statement about a *sample* of instances. The MILP supplies
something stronger:

In [6]:
print(f"MILP returned status OPTIMAL on {int(df.milp_proven.sum())} of {len(df)} instances.")
print(f"GA proved optimality on 0 of {len(df)} — no metaheuristic can.")

MILP returned status OPTIMAL on 20 of 20 instances.
GA proved optimality on 0 of 20 — no metaheuristic can.


`OPTIMAL` is a **certificate**: the solver has proven, via the LP relaxation bound,
that no cheaper feasible assignment exists. It is a deductive guarantee about *this*
instance, not an inductive claim from repeated runs.

A GA can only ever report the best solution it happened to find. It cannot say
whether that is the optimum or 30% away from it — which, at n = 120, is exactly the
error it is making without any way to detect it.

Three consequences for this system:

1. **Bounded regret.** A warehouse manager approving a slotting plan is approving a
   provably cost-minimal one, not one that "looks reasonable".
2. **Reproducibility.** The MILP returns the same assignment on every run
   (objective range 0.0 across repeated solves). Under GDPR Article 22, an automated
   decision must be explicable and reproducible; a solver whose output varies by seed
   makes both harder to satisfy.
3. **Explanation.** Because the objective decomposes into named cost terms, each
   assignment can be explained by the margin over the runner-up — which is what the
   slotting screen now shows per line. A GA fitness score does not decompose this way.

## 6. Threats to validity

Stated rather than omitted:

- **The GA is our implementation.** A differently tuned GA, or a memetic variant with
  local search, would narrow the gap. The claim is not that no GA can do better; it is
  that a standard, fairly resourced GA does not, and cannot certify its answer.
- **Instances are subsets of one synthetic population.** External validity beyond this
  warehouse is `UNVERIFIED`, consistent with the model card.
- **The relocation cap was left unconstrained** so the comparison isolates search
  quality. A binding cap makes the problem harder for both methods.
- **Sizes stop at 120** because the population holds 144 materials. Extrapolating the
  trend past that range is not supported by this evidence.

In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
g = df.groupby("size")

ax[0].errorbar(g.gap_best_pct.mean().index, g.gap_best_pct.mean().values,
               yerr=g.gap_best_pct.std().values, marker="o", capsize=4,
               color="#C2185B", label="GA best of 5")
ax[0].errorbar(g.gap_mean_pct.mean().index, g.gap_mean_pct.mean().values,
               yerr=g.gap_mean_pct.std().values, marker="s", capsize=4,
               color="#7B1FA2", alpha=.65, label="GA mean of 5")
ax[0].axhline(0, color="#333", lw=1.2, ls="--", label="MILP (proven optimal)")
ax[0].set_xlabel("materials per instance"); ax[0].set_ylabel("optimality gap (%)")
ax[0].set_title("GA cost above the proven optimum"); ax[0].legend(frameon=False, fontsize=9)
ax[0].grid(alpha=.25)

ax[1].scatter(df["size"], df.milp_seconds*1000, color="#00695C", label="MILP solve")
ax[1].scatter(df["size"], df.ga_budget*1000, color="#BDBDBD", marker="_", s=200,
              label="GA budget granted")
ax[1].set_yscale("log"); ax[1].set_xlabel("materials per instance")
ax[1].set_ylabel("milliseconds (log)"); ax[1].set_title("Compute: MILP vs GA budget")
ax[1].legend(frameon=False, fontsize=9); ax[1].grid(alpha=.25, which="both")

fig.tight_layout()
fig.savefig("outputs/solver_evidence/milp_vs_ga.png", dpi=150)
print("figure -> outputs/solver_evidence/milp_vs_ga.png")
fig

figure -> outputs/solver_evidence/milp_vs_ga.png


<Figure size 1200x420 with 2 Axes>

## 7. Conclusion

MILP was selected for slotting on three grounds, in order of weight:

1. **It certifies optimality.** On every instance tested it returned `OPTIMAL`, a
   proof rather than an observation. The GA cannot do this at any size.
2. **The quality gap grows with scale** (rho = +0.954, p = 7.5e-11), reaching
   **+26.4%** at 120 materials while the GA held a 138× compute advantage. The real
   warehouse operates at the top of this range.
3. **It is deterministic**, which reproducibility, auditability and the
   per-line explanations all depend on.

The honest boundary: **below roughly 40 materials the two are equivalent**, and at
n = 20 the confidence interval includes zero. Had this warehouse been that small, the
choice would not have been justified on these grounds.